# ⚡ Google Colab TPU v5e High-Performance AI Runner (JAX / PyTorch-TPU)
Этот блокнот оптимизирован специально под **Google Cloud TPU v5e** (197 TFLOPS, 16 GB HBM, аппаратные матричные движки bfloat16).

### 💡 Преимущества TPU v5e:
1. Пропускная способность памяти HBM **810 ГБ/с** (более чем в 2.5 раза быстрее, чем GDDR6 у T4 GPU).
2. Нативная аппаратная поддержка формата **bfloat16** (нет артефактов и потерь качества от 4-битного квантования).
3. Доступны два способа работы:
   - **PyTorch (`torch-tpu`)**: стандартные модели Hugging Face через `device_map='tpu'`.
   - **Google JAX / KerasHub**: максимальная скорость через XLA-компиляцию прямо в матричные инструкции TPU.


In [ ]:
# 1. Диагностика аппаратного ускорителя TPU v5e
import os
import subprocess

print("--- Статус TPU устройств ---")
# Проверка устройств JAX
try:
    import jax
    print("JAX Backend:", jax.default_backend())
    print("JAX Devices:", jax.devices())
except Exception as e:
    print("JAX init:", e)

# Проверка PyTorch Accelerator
try:
    import torch
    if hasattr(torch, "accelerator") and torch.accelerator.is_available():
        print(f"PyTorch Accelerator: {torch.accelerator.current_accelerator().type}")
        print(f"Количество ядер: {torch.accelerator.device_count()}")
    else:
        import torch_xla.core.xla_model as xm
        print(f"PyTorch XLA Device: {xm.xla_device()}")
except Exception as e:
    print("PyTorch TPU check:", e)


In [ ]:
# 2. Установка оптимизированного стека для TPU v5e
!pip install -q -U torch-tpu transformers accelerate gradio
!pip install -q -U keras keras-hub


In [ ]:
# 3. Запуск языковой модели на TPU v5e (Hugging Face PyTorch bfloat16)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Быстрая и качественная модель (Qwen 2.5 или Llama 3.2)
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
print(f"Загрузка {MODEL_ID} на TPU v5e в нативном формате bfloat16...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

try:
    # Прямая загрузка в память TPU через torch-tpu
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="tpu"
    )
    print("Модель успешно размещена на TPU v5e!")
except Exception as e:
    print(f"Fallback на torch_xla: {e}")
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16
    ).to(device)
    print(f"Модель размещена на XLA устройстве: {device}")

# Тестовая генерация
prompt = "Объясни в 3 предложениях, почему архитектура TPU эффективна для нейросетей."
messages = [{"role": "user", "content": prompt}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        inputs.to(model.device),
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7
    )

response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
print("\nОтвет модели:\n", response)


In [ ]:
# 4. [Альтернатива] Максимальный Throughput через Google KerasHub (JAX XLA)
import os
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.0"

import keras_hub

print("Загрузка модели через KerasHub с XLA-компиляцией под TPU...")
# Модель Gemma 2 от Google нативно оптимизирована для TPU v5e
gemma_lm = keras_hub.models.CausalLM.from_preset("hf://google/gemma-2-2b-it")

output = gemma_lm.generate("Привет! Расскажи о преимуществах архитектуры TPU v5e.", max_length=150)
print("\nОтвет Gemma на JAX:\n", output)


In [ ]:
# 5. Интерактивный чат-интерфейс Gradio
import gradio as gr

def chat_fn(message, history):
    messages = []
    for h in history:
        messages.append({"role": "user", "content": h[0]})
        messages.append({"role": "assistant", "content": h[1]})
    messages.append({"role": "user", "content": message})

    prompt_inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(prompt_inputs, max_new_tokens=256, temperature=0.7)
    answer = tokenizer.decode(out[0][len(prompt_inputs[0]):], skip_special_tokens=True)
    return answer

demo = gr.ChatInterface(
    fn=chat_fn,
    title=f"Colab TPU v5e AI Assistant ({MODEL_ID})",
    description="Высокопроизводительный инференс на Google Cloud TPU v5e (16 GB HBM)"
)

demo.launch(share=True, debug=False)


### 🔌 Опционально: SSH-подключение к TPU v5e ноде с вашего локального ПК

In [ ]:
# ==============================================================================
# Google Colab SSH Bridge (colab-ssh + Cloudflare Tunnel)
# ==============================================================================
!pip install -q colab_ssh --upgrade

from colab_ssh import launch_ssh_cloudflared

# Пароль для входа (на случай ручного ввода)
PASSWORD = "colab"

# Запуск туннеля через официальную библиотеку colab-ssh
launch_ssh_cloudflared(password=PASSWORD)

# Авторизация открытого ключа вашего ПК для мгновенного входа без пароля
LOCAL_PUBKEY = "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIAlcVVcs5DNr2kNfwLJtBjYQocNsdLCDcUKRIgquwK9J colab-bridge@local"
!mkdir -p /root/.ssh && echo "{LOCAL_PUBKEY}" >> /root/.ssh/authorized_keys && chmod 600 /root/.ssh/authorized_keys
